In [22]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:

import os

os.listdir("/content/drive/MyDrive")

['Colab Notebooks',
 'Queue',
 'AAt2',
 'Certificate ',
 'toc',
 'resume ',
 'AMBIKA RESUME.pdf',
 'n8n',
 'deep learning']

In [24]:
import pandas as pd
from transformers import T5Tokenizer,Trainer, TrainingArguments , T5ForConditionalGeneration

train = pd.read_csv(
    "/content/drive/MyDrive/deep learning/samsum-train.csv"
)

val_data = pd.read_csv(
    "/content/drive/MyDrive/deep learning/samsum-validation.csv"
)

In [25]:
train.head()
train["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [26]:
train.shape

(14732, 3)

In [27]:
val_data.shape

(818, 3)

In [28]:
train_data= train.sample(n=4000,random_state=42).reset_index(drop=True)
val_data= val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [29]:
train.shape

(14732, 3)

# *Data Pre-Processing*

In [30]:
import re

def clean_data(text):
  text=re.sub(r"\r\n"," ",text) #lines
  text=re.sub(r"\s+"," ",text) # spaces
  text=re.sub(r"<.*?>"," ",text) # hmtl
  text=text.strip().lower()

  return text

In [31]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)


In [32]:
val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

# Tokenization

In [33]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [34]:
def tokenize(data):
    inputs= tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    target= tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"]=target["input_ids"] # token ids => add input to labels
    return inputs

In [35]:
train_dataset= train_data.apply(tokenize,axis=1).tolist()
val_dataset= val_data.apply(tokenize,axis=1).tolist()

In [36]:
train_dataset[0]
# input ids = token ids & end "1" => End of seq and 0's are paddings
#attention mask=> vaild tokens len sm as list

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [37]:
len(train_dataset[0]["input_ids"])

512

In [38]:
type(train_dataset)
type(val_dataset)

list

# Fine tuning With Model

In [39]:
model= T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [40]:
#set up device
import torch

if torch.backends.mps.is_available():
    device= torch.device("mps")
elif torch.cuda.is_available():
    device= torch.device("cuda")
else:
    device= torch.device("cpu")

print("device:",device)
model.to(device)

device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [41]:
#create training arguments

training_args= TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500
)

In [42]:
#define trainer class

trainer= Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

training the model

In [43]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.647178,0.381366
2,0.396711,0.359593
3,0.373344,0.354282
4,0.361063,0.350198
5,0.354967,0.349167
6,0.350882,0.348717


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9140240732828776, metrics={'train_runtime': 1235.9926, 'train_samples_per_second': 19.418, 'train_steps_per_second': 2.427, 'total_flos': 3248203235328000.0, 'train_loss': 0.9140240732828776, 'epoch': 6.0})

## Saving the model


In [44]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_tokenizer/tokenizer_config.json',
 './saved_summary_tokenizer/tokenizer.json')

### load saved model

In [45]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_tokenizer")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Tesing the core logic of the model



In [60]:
def summarizer_dialogue(dialogue):

  #preprocessing
  dialogue=clean_data(dialogue)

  #tokenize
  inputs=tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt" # default hugging face model be in pytorch form
  ).to(device)

  #model generate the summary --> token id's
  model.to(device)
  targets=model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  #decoded
  summary= tokenizer.decode(targets[0], skip_special_tokens=True)
  return summary

In [61]:
test_dialogue="""**A:** Hey! What do you think about AI becoming so common nowadays?

**B:** I think it’s really useful. It can help us learn faster and automate boring tasks.

**A:** True! But don’t you think AI might replace some jobs?

**B:** Some jobs, probably. But I think people who learn to use AI will have more opportunities.

**A:** That makes sense. So instead of being afraid of AI, we should learn how to work with it.

**B:** Exactly! AI is a tool. How we use it matters more than the technology itself."""



In [62]:
summary=summarizer_dialogue(test_dialogue)
print("Summary:",summary)

Summary: ai is becoming so common nowadays. it can help us learn faster and automate boring tasks.
